In [ ]:
import requests
import lxml.html
import pandas as pd
import os
import us
import jellyfish

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
from pathlib import Path
from google.genai import types
from pypdf import PdfReader
from datetime import datetime

In [4]:
from ingest_sc_cases import case_df

In [5]:
from ingest_sc_judges import judge_pd

In [62]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,docket_no,title,state,date,type,pending,opinion_link
13,2026-00384,Williams v. Board of Elections of the State of...,New York,"March 19, 2026",Voting Rights and Elections,False,None
38,20220896,State v. Andrus,Utah,"August 7, 2025",Criminal Law,False,None
50,S-1-SC-40715,Franklin v. Martinez,New Mexico,"January 26, 2026",Criminal Law,False,None
98,22 OC 00022 1 B,Persaud-Zamora v. Cegavske,Nevada,"April 26, 2022",Voting Rights and Elections,False,None
102,EF2025-2536,Texas v. Bruck,Texas,"October 31, 2025",Reproductive Rights,False,None
104,2025AP002121,Voces de la Frontera v. Gerber,Wisconsin,"September 18, 2025",Government Structure,False,None
121,PR-123179,McVay v. Cockroft,Oklahoma,"July 28, 2025",Voting Rights and Elections,False,None
153,15-25-00093-CV,State v. City of San Antonio,Texas,"June 19, 2025",Reproductive Rights,False,None
207,CV01-23-14744,Adkins v. State,Idaho,"April 11, 2025",Reproductive Rights,False,None
229,SJC-13666,Gotay v. Creen,Massachusetts,"March 21, 2025",Civil Due Process,False,None


In [48]:
class Opinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    opinion_summary: str = Field(
        description="An extremely brief description of the judge's opinion on the case."
    )


class Politics(BaseModel):
    dim_envir: bool = Field(description="Protects the environment")
    dim_consum: bool = Field(description="Protects consumers")
    dim_reprights: bool = Field(description="Protects reproductive rights")
    dim_demnorms: bool = Field(description="Protects democratic norms")
    dim_press: bool = Field(description="Protects the press")
    dim_pubhealth: bool = Field(description="Protects public health")
    dim_church: bool = Field(description="Protects the separation of church and state")
    dim_voting: bool = Field(description="Protects voting access")
    dim_pubedu: bool = Field(description="Protects public education")
    dim_civlib: bool = Field(description="Protects civil liberties")
    dim_priv: bool = Field(description="Protects privacy")
    dim_workers: bool = Field(description="Protects workers")


class Case(BaseModel):
    title: str = Field(description="The title of the case.")
    decision: str = Field(description="The final decision that was made for the case.")
    issue_debate: str = Field(description="The main issue being debated in the case.")
    plaintiff_issues: Politics
    judge_opinions: List[Opinion]

In [ ]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    """
    Inputs:
    - pdf_link: string
    - model_id: string
    - client: genai.Client
    - prompt: str

    Outputs:
    - dict

    Function makes a request to the content from a pdf url and prompts it into
    Gemini through the inputted gemini client. Returns a dictionary following
    the structure of the Case class defined outside of the function.
    """
    # Make request to content
    resp = requests.get(pdf_link).content

    # Prompt Gemini
    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text).model_dump()

In [ ]:
def analyze_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    """
    Inputs:
    - case_df: pd.Dataframe,
    - judge_df: pd.Dataframe,
    - prompt_start: str,
    - client_info: dict

    Outputs:
    - file_dic: dict

    Function creates the full prompt by taking the prompt start (extracted)
    from .txt file and joins it with the string version of the judge
    dataframe. It then extract the information about the client, the model id
    and gemini client.
    """
    # Create full prompt with judge dataframe
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)

    # Extract Gemini client information
    model_id = client_info["model_id"]
    client = client_info["client"]

    # Initialize return dictionary
    num_cases = len(case_df)
    file_dic = {}
    for i in range(num_cases):
        # Create case id with docket number, state, and date
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_ref = "_".join([docket_no, state, date])

        # Extract pdf link and prompt it to Gemini using read_opinion()
        pdf_link = case_df.iloc[i]["opinion_link"]
        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        # Fill return dictionary with a dictionary for each case containing
        # the pdf link and the Gemini response
        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp

    return file_dic

In [ ]:
def apply_model(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    """
    Inputs:
    - case_df: pd.Dataframe
    - judge_df: pd.Dataframe
    - prompt_path: str

    Outputs:
    - case_dic: dict

    This function extracts the fixed part of the prompt by reading the .txt
    file, and it initializes the Gemini model by using the API key. It then
    calls analyze_cases() returns its output.
    """
    # Extract the prompt start from the .txt file
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()

    # Find API key in directory (should be in a .env file)
    load_dotenv(find_dotenv())

    # Initialize gemini client
    gemini_key = os.getenv("GEMINI_API_KEY")
    client_info = {"model_id": "gemini-2.5-flash-lite", "client": genai.Client(api_key=gemini_key)}

    #
    case_dic = analyze_cases(case_df, judge_df, prompt_start, client_info)

    return case_dic

In [ ]:
def state_opinion_table(case_dic: dict):
    """
    Inputs:
    - case_dic: dict (the output of apply_model())

    Outputs:
    - pd.DataFrame

    This function converst the output from apply_model() into an opinion
    table with three columns:
    - case_id
    - name
    - opinion
    """
    opinion_table = {"case_id": [], "name": [], "opinion": []}

    for key in case_dic.keys():
        opinions = case_dic[key]["response"]
        num_opinions = len(opinions["judge_opinions"])

        for i in range(num_opinions):
            opinion_table["case_id"].append(key)
            judge_name = opinions["judge_opinions"][i]["judge_name"]
            opinion_table["name"].append(judge_name)
            opinion_summary = opinions["judge_opinions"][i]["opinion_summary"]
            opinion_table["opinion"].append(opinion_summary)

    return pd.DataFrame(opinion_table)

In [ ]:
def state_case_table(case_df: pd.DataFrame, case_dic: dict):
    """
    Inputs:
    - case_df: pd.DataFrame
    - case_dic: dict

    Outputs:
    - pd.DataFrame

    This function takes a dataframe of cases, and compines it with the output
    of apply_model() to create a complete case table that includes case
    decisions and political dimensions.
    """
    case_table = {
        "case_id": [],
        "docket_no": [],
        "title": [],
        "state": [],
        "date": [],
        "type": [],
        "dispute_desc": [],
        "decision_outcome": [],
        "decision_status": [],
        "dim_envir": [],
        "dim_consum": [],
        "dim_reprights": [],
        "dim_demnorms": [],
        "dim_press": [],
        "dim_pubhealth": [],
        "dim_church": [],
        "dim_voting": [],
        "dim_pubedu": [],
        "dim_civlib": [],
        "dim_priv": [],
        "dim_workers": [],
    }
    num_cases = len(case_df)

    for i in range(num_cases):
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        case_table["docket_no"].append(docket_no)
        state = case_df.iloc[i]["state"]
        case_table["state"].append(state)
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_table["date"].append(date)

        title = case_df.iloc[i]["title"]
        case_table["title"].append(title)
        type = case_df.iloc[i]["type"]
        case_table["type"].append(type)

        case_id = "_".join([docket_no, state, date])
        case_table["case_id"].append(case_id)

        status = case_df.iloc[i]["pending"]
        case_table["decision_status"].append(status)

        response = case_dic[case_id]["response"]
        case_table["dispute_desc"].append(response["issue_debate"])
        case_table["decision_outcome"].append(response["decision"])

        politics = response["plaintiff_issues"]
        for key in politics.keys():
            case_table[key].append(politics[key])

    return pd.DataFrame(case_table)

In [ ]:
def produce_tables(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    """
    Inputs:
    - case_df: pd.DataFrame
    - judge_df: pd.DataFrame
    - prompt_path: str

    Outputs:
    - rd: dict

    This function takes dataframes of cases and judges, and iterates state
    by state to iteratively create the opinion and case tables. Returns
    each in a dictionary.
    """
    # Extracting the unique states from the case dataframe
    states = case_df["state"].sort_values().unique()

    opinions = []
    cases = []

    for state in states:
        # Extract every case for that state
        state_cases = case_df[case_df["state"] == state]

        # Extract every judge for that state using the state's abbreviation
        state_abbr = str(us.states.lookup(state).abbr)
        state_judges = judge_df[judge_df["state"] == state_abbr]

        # Apply the Gemini prompt to the cases and judges
        case_dic = apply_model(state_cases, state_judges, prompt_path)

        # Append case and opinion tables
        opinions.append(state_opinion_table(case_dic))
        cases.append(state_case_table(state_cases, case_dic))

    rd = {"opinion_table": pd.concat(opinions), "case_table": pd.concat(cases)}

    return rd

In [65]:
ala_cases = case_df[case_df["state"] == "Alaska"].head(5)
ala_judges = judge_pd[judge_pd["state"] == "AK"]

alaska = produce_tables(ala_cases, ala_judges, "prompt_1.txt")

Querying case 0
Querying case 1
Querying case 2
Querying case 3
Querying case 4


In [66]:
display(alaska["opinion_table"])
display(alaska["case_table"])

,case_id,name,opinion
0,S-18210-_Alaska_2022/10/21,dario borghesan,Authored the opinion explaining the court's re...
1,S-18210-_Alaska_2022/10/21,jennifer s. henderson,Was part of the panel that affirmed the superi...
2,S-18210-_Alaska_2022/10/21,susan m. carney,Was part of the panel that affirmed the superi...
3,"S-19083,-S-19113_Alaska_2025/03/28",dario borghesan,"Authored the opinion of the court, finding the..."
4,"S-19083,-S-19113_Alaska_2025/03/28",jennifer s. henderson,Concurred with the majority opinion.
5,"S-19083,-S-19113_Alaska_2025/03/28",jude pate,Concurred with the majority opinion.
6,S-18814_Alaska_2025/01/17,Dario Borghesan,"Justice Borghesan authored the opinion, conclu..."
7,S-18814_Alaska_2025/01/17,Susan M. Carney,Justice Carney concurred with the decision.
8,S-18814_Alaska_2025/01/17,Jennifer S. Henderson,Justice Henderson concurred with the decision.
9,S-18814_Alaska_2025/01/17,Jude Pate,Justice Pate concurred with the decision.


,case_id,docket_no,title,state,date,type,dispute_desc,decision_outcome,decision_status,dim_envir,...,dim_reprights,dim_demnorms,dim_press,dim_pubhealth,dim_church,dim_voting,dim_pubedu,dim_civlib,dim_priv,dim_workers
0,S-18210-_Alaska_2022/10/21,S-18210-,Kohlhaas v. State,Alaska,2022/10/21,Voting Rights and Elections,The case debated whether Alaska's Initiative 2...,The Supreme Court affirmed the superior court'...,False,False,...,False,True,False,False,False,True,False,False,False,False
1,"S-19083,-S-19113_Alaska_2025/03/28","S-19083,-S-19113",State Department of Education & Early Developm...,Alaska,2025/03/28,Criminal Law,The main issue debated was whether Alaska stat...,The Supreme Court reversed the lower court's d...,False,False,...,False,False,False,False,False,False,True,False,False,False
2,S-18814_Alaska_2025/01/17,S-18814,Valoaga v. State,Alaska,2025/01/17,Criminal Law,The main issue debated is whether the Alaska C...,The Supreme Court of Alaska affirmed the super...,False,False,...,False,False,False,False,False,False,False,True,False,False
3,S-18189_Alaska_2024/05/10,S-18189,Alaska Trappers Association v. City of Valdez,Alaska,2024/05/10,Government Structure,The main issue debated is whether the City of ...,The court affirmed the superior court's grant ...,False,True,...,False,False,False,False,False,False,False,False,False,False
4,S-17910_Alaska_2024/03/08,S-17910,State v. McKelvey,Alaska,2024/03/08,Civil Rights,The main issue is whether law enforcement's us...,The Alaska Supreme Court affirmed the Court of...,False,False,...,False,False,False,False,False,False,False,False,True,False


In [36]:
def find_match(name_str: str, name_list: list):
    sim_scores = {"alias": name_list, "sim_score": []}

    for alias in name_list:
        sim_score = jellyfish.jaro_winkler_similarity(name_str, alias)
        sim_scores["sim_score"].append(sim_score)

    return pd.DataFrame(sim_scores).sort_values(by="sim_score", ascending=False).iloc[0]["alias"]

In [38]:
def standardize_names(opinion_table: pd.DataFrame, judge_df: pd.DataFrame):
    r_table = opinion_table.copy()
    known_names = list(judge_df["name"])

    for i, j_name in enumerate(opinion_table["name"]):
        alias = find_match(j_name, known_names)
        r_table.loc[i, "name"] = alias

    return r_table

In [67]:
standardize_names(alaska["opinion_table"], ala_judges)

,case_id,name,opinion
0,S-18210-_Alaska_2022/10/21,Dario Borghesan,Authored the opinion explaining the court's re...
1,S-18210-_Alaska_2022/10/21,Jennifer S. Henderson,Was part of the panel that affirmed the superi...
2,S-18210-_Alaska_2022/10/21,Susan M. Carney,Was part of the panel that affirmed the superi...
3,"S-19083,-S-19113_Alaska_2025/03/28",Dario Borghesan,"Authored the opinion of the court, finding the..."
4,"S-19083,-S-19113_Alaska_2025/03/28",Jennifer S. Henderson,Concurred with the majority opinion.
5,"S-19083,-S-19113_Alaska_2025/03/28",Jude Pate,Concurred with the majority opinion.
6,S-18814_Alaska_2025/01/17,Dario Borghesan,"Justice Borghesan authored the opinion, conclu..."
7,S-18814_Alaska_2025/01/17,Susan M. Carney,Justice Carney concurred with the decision.
8,S-18814_Alaska_2025/01/17,Jennifer S. Henderson,Justice Henderson concurred with the decision.
9,S-18814_Alaska_2025/01/17,Jude Pate,Justice Pate concurred with the decision.
